# TidalSense N-Tidal Clinical Study
## Part 2: Fault Investigation

This notebook investigates issues found in the study data.
For each fault we describe what was found, what likely caused it, and what should be done.

Faults covered:
1. PRC-007 has an abnormally high error rate
2. Handsets 13 and 14 did not receive firmware 2.0.1
3. Handset-00005 stopped sending hardware telemetry mid-study
4. PRC-009 is running tests on weekends
5. CRITICAL hardware alerts on specific handsets
6. Three firmware log entries have a missing version value

## Step 1: Load the Data

In [ ]:
import pandas as pd
import json

sw = pd.read_csv('software_table.csv')
hw = pd.read_csv('hardware_telemetry_table.csv')

sw['test_log_time'] = pd.to_datetime(sw['test_log_time'])
sw['test_processed_time'] = pd.to_datetime(sw['test_processed_time'])
hw['log_time'] = pd.to_datetime(hw['log_time'])

print('Software table rows:', len(sw))
print('Hardware table rows:', len(hw))

## Step 2: Data Quality Checks and Cleaning

Before any analysis we check both tables for quality issues.
The software table was clean. The hardware table had four issues that needed fixing.

In [ ]:
# Check for missing values in both tables
print('Missing values in software table:')
print(sw.isnull().sum())
print()
print('Missing values in hardware table:')
print(hw.isnull().sum())

In [ ]:
# Check for duplicate rows in the hardware table
dup_cols = ['ID', 'boot_ID', 'log_time', 'component', 'message']
duplicate_count = hw.duplicated(subset=dup_cols).sum()

print('Exact duplicate rows found:', duplicate_count)
print()
print('Sample of duplicates (same handset, same timestamp, same reading):')
print(hw[hw.duplicated(subset=dup_cols, keep=False)][['ID', 'log_time', 'component', 'message']].head(4).to_string())

In [ ]:
# Fix 1: Remove exact duplicate rows
# These are the same reading sent twice by the telemetry pipeline, not real double readings
hw = hw.drop_duplicates(subset=dup_cols)
print('Rows after removing duplicates:', len(hw))

In [ ]:
# Fix 2: Drop the 8 rows where the message field is completely empty
# These have a component label but no data to parse, so they are not usable
hw = hw[hw['message'].notnull()]
print('Rows after dropping empty messages:', len(hw))

In [ ]:
# Parse the JSON message column into a dictionary
# This is needed before we can recover the null component rows
def parse_message(msg):
    try:
        return json.loads(msg)
    except:
        return {}

hw['parsed'] = hw['message'].apply(parse_message)

In [ ]:
# Fix 3: Recover the 102 rows where the component column is null
# The message content tells us what each one is even without the label

def infer_component(parsed_msg):
    if 'voltage' in parsed_msg:
        return 'battery'
    elif 'signal_strength' in parsed_msg:
        return 'modem'
    elif 'firmware_version' in parsed_msg:
        return 'firmware'
    else:
        return None

null_comp_mask = hw['component'].isnull()
hw.loc[null_comp_mask, 'component'] = hw.loc[null_comp_mask, 'parsed'].apply(infer_component)

print('Null component rows remaining after recovery:', hw['component'].isnull().sum())

In [ ]:
# Summary of what was cleaned and one important note about the status field
print('Hardware table cleaning summary:')
print('  Original rows:                  3,426')
print('  Duplicate rows removed:            33')
print('  Empty message rows dropped:         8')
print('  Null component rows recovered:    102')
print('  Final usable rows:             ', len(hw))
print()
print('Note on status field:')
print('Most battery and modem readings have an empty string in the status field')
print('rather than a label like OK or CRITICAL. Because of this the fault analysis')
print('uses voltage and signal values directly rather than the status field alone.')

## Step 3: Add Helper Columns and Split Hardware Table

In [ ]:
# Helper columns for the software table
sw['date'] = sw['test_log_time'].dt.date
sw['weekday'] = sw['test_log_time'].dt.day_name()
sw['is_weekend'] = sw['weekday'].isin(['Saturday', 'Sunday'])
sw['any_error'] = (
    sw['automated_check_hardware_error'] |
    sw['automated_check_software_error'] |
    (sw['user_error'] != 'no_error') |
    sw['miscalibration']
)

# Split hardware table into three component tables
fw = hw[hw['component'] == 'firmware'].copy()
fw['version'] = fw['parsed'].apply(lambda x: x.get('firmware_version', None))

modem = hw[hw['component'] == 'modem'].copy()
modem['status'] = modem['parsed'].apply(lambda x: x.get('status', ''))
modem['signal'] = modem['parsed'].apply(lambda x: x.get('signal_strength', None))

battery = hw[hw['component'] == 'battery'].copy()
battery['status'] = battery['parsed'].apply(lambda x: x.get('status', ''))
battery['voltage'] = battery['parsed'].apply(lambda x: x.get('voltage', None))

print('Firmware rows:', len(fw))
print('Modem rows:   ', len(modem))
print('Battery rows: ', len(battery))

---
## FAULT 1: PRC-007 Has an Abnormally High Error Rate

In [ ]:
prc7 = sw[sw['practice_ID'] == 'PRC-007']
others = sw[sw['practice_ID'] != 'PRC-007']

print('PRC-007 total tests:', len(prc7))
print('All other practices total tests:', len(others))

In [ ]:
# Compare each error type between PRC-007 and the rest
prc7_hw_err  = round(prc7['automated_check_hardware_error'].mean() * 100, 1)
prc7_sw_err  = round(prc7['automated_check_software_error'].mean() * 100, 1)
prc7_miscal  = round(prc7['miscalibration'].mean() * 100, 1)
prc7_any     = round(prc7['any_error'].mean() * 100, 1)

other_hw_err = round(others['automated_check_hardware_error'].mean() * 100, 1)
other_sw_err = round(others['automated_check_software_error'].mean() * 100, 1)
other_miscal = round(others['miscalibration'].mean() * 100, 1)
other_any    = round(others['any_error'].mean() * 100, 1)

print('Error type         PRC-007    Others')
print('Any error          ', prc7_any, '%      ', other_any, '%')
print('Hardware error     ', prc7_hw_err, '%       ', other_hw_err, '%')
print('Software error     ', prc7_sw_err, '%       ', other_sw_err, '%')
print('Miscalibration     ', prc7_miscal, '%      ', other_miscal, '%')

In [ ]:
print('Handsets used by PRC-007:')
print(prc7['handset_ID'].unique())

In [ ]:
# Check if error rate changed after firmware 2.0.1 release on 28 April
update_date = pd.Timestamp('2026-04-28')

prc7_before = prc7[prc7['test_log_time'] < update_date]
prc7_after  = prc7[prc7['test_log_time'] >= update_date]

print('PRC-007 miscalibration rate before 28 Apr:', round(prc7_before['miscalibration'].mean() * 100, 1), '%')
print('PRC-007 miscalibration rate after  28 Apr:', round(prc7_after['miscalibration'].mean() * 100, 1), '%')
print()
print('Rate worsened after the update date, consistent with these handsets')
print('not receiving the update that improved other handsets.')

---
## FAULT 2: Handsets 13 and 14 Did Not Receive Firmware 2.0.1

In [ ]:
# Most recent firmware per handset
fw_sorted = fw.sort_values('log_time')
latest_fw = fw_sorted.groupby('ID').last()[['version', 'log_time']]

print('Latest firmware version seen per handset:')
print(latest_fw.to_string())

In [ ]:
# Which handsets received 2.0.1 and when
fw_201 = fw[fw['version'] == '2.0.1']
first_201 = fw_201.groupby('ID')['log_time'].min().sort_values()

print('Handsets that received 2.0.1 and when:')
print(first_201.to_string())
print()
print('Handsets 13 and 14 are not in this list.')

In [ ]:
# Full firmware history for handsets 13 and 14
fw_1314 = fw[fw['ID'].isin([13, 14])].sort_values('log_time')

print('Full firmware timeline for handsets 13 and 14:')
print(fw_1314[['ID', 'log_time', 'version']].to_string())

---
## FAULT 3: Handset-00005 Stopped Sending Hardware Telemetry Mid-Study

In [ ]:
h5_sw = sw[sw['handset_ID'] == 'handset-00005']
h5_hw = hw[hw['ID'] == 5]

print('Software table entries for handset-00005:')
print('  Total tests:  ', len(h5_sw))
print('  Date range:   ', h5_sw['test_log_time'].min().date(), 'to', h5_sw['test_log_time'].max().date())
print('  Boot IDs seen:', sorted(h5_sw['boot_ID'].unique()))
print()
print('Hardware table entries for handset-00005 (ID=5):')
print('  Total logs:   ', len(h5_hw))
print('  Date range:   ', h5_hw['log_time'].min().date(), 'to', h5_hw['log_time'].max().date())
print('  Boot IDs seen:', sorted(h5_hw['boot_ID'].unique()))

In [ ]:
# Tests run after telemetry stopped
cutoff = pd.Timestamp('2026-04-15')
h5_after = h5_sw[h5_sw['test_log_time'] > cutoff]

print('Tests run after telemetry stopped (after 15 Apr):', len(h5_after))
print('These tests have no hardware health data to support them.')

In [ ]:
# Additional finding: handset 5 logged telemetry outside working hours
# All other handsets transmit only during normal hours (roughly 07:00 to 19:00)
h5_hw = h5_hw.copy()
h5_hw['hour'] = h5_hw['log_time'].dt.hour
out_of_hours = h5_hw[(h5_hw['hour'] < 7) | (h5_hw['hour'] >= 19)]

print('Handset-00005 telemetry entries outside 07:00 to 19:00:')
print(out_of_hours[['log_time', 'component']].to_string())
print()
print('No other handset has entries at these times.')
print('Possible causes: device left on overnight, clock drift, or cached logs replaying.')

In [ ]:
# Last known firmware for handset 5
h5_fw = fw[fw['ID'] == 5]

print('Firmware records for handset-00005:')
print(h5_fw[['log_time', 'version']].to_string())
print()
print('Last known firmware: 1.0.0 as of 9 April.')
print('Cannot confirm whether any later versions were received.')

---
## FAULT 4: PRC-009 Is Running Tests on Weekends

In [ ]:
weekend_tests = sw[sw['is_weekend'] == True]

print('Total weekend tests:', len(weekend_tests))
print()
print('Weekend tests by practice:')
print(weekend_tests.groupby('practice_ID').size())

In [ ]:
prc9_weekend = weekend_tests[weekend_tests['practice_ID'] == 'PRC-009']

print('Weekend dates and counts for PRC-009:')
print(prc9_weekend.groupby(['date', 'weekday']).size().to_string())
print()
print('Handsets used on weekends:')
print(prc9_weekend['handset_ID'].value_counts())
print()
print('Administrators running weekend tests:')
print(prc9_weekend['administrator_ID'].value_counts())

In [ ]:
# Check if the same admins also run weekday tests at PRC-009
prc9_weekday = sw[(sw['practice_ID'] == 'PRC-009') & (sw['is_weekend'] == False)]

print('Admins running weekday tests at PRC-009:')
print(prc9_weekday['administrator_ID'].value_counts())

In [ ]:
# Show PRC-009 volume vs peers with weekends removed
weekday_only = sw[sw['is_weekend'] == False]

print('Tests per practice (weekdays only):')
print(weekday_only.groupby('practice_ID').size().sort_values(ascending=False))

---
## FAULT 5: CRITICAL Hardware Alerts on Specific Handsets

In [ ]:
# Note: most readings have an empty string status rather than OK
# so we filter specifically for CRITICAL rather than assuming empty means healthy
crit_modem = modem[modem['status'] == 'CRITICAL']

print('CRITICAL modem alerts per handset:')
print(crit_modem.groupby('ID').size().sort_values(ascending=False))

In [ ]:
print('Signal strength at CRITICAL modem events (by handset):')
print(crit_modem.groupby('ID')['signal'].describe().round(2))
print()
print('Average signal strength across all handsets:', round(modem['signal'].mean(), 2))

In [ ]:
crit_battery = battery[battery['status'] == 'CRITICAL']

print('CRITICAL battery alerts per handset:')
print(crit_battery.groupby('ID').size().sort_values(ascending=False))

In [ ]:
# Voltage at CRITICAL events for handset 13
# Also show the five lowest voltages in the dataset to highlight the declining trend
crit_bat_13 = crit_battery[crit_battery['ID'] == 13]
print('Voltage at CRITICAL battery events for handset 13:')
print(crit_bat_13[['log_time', 'voltage']].to_string())

print()
print('Five lowest battery voltages in the entire dataset:')
print(battery.nsmallest(5, 'voltage')[['ID', 'log_time', 'voltage']].to_string())
print()
print('All five lowest readings are from handset 13 in the final two weeks.')
print('The battery is actively deteriorating, not just sitting at a stable low level.')

In [ ]:
# The single clearest number showing the hardware problem at PRC-007
bat_1314   = battery[battery['ID'].isin([13, 14])]
bat_others = battery[~battery['ID'].isin([13, 14])]

print('Average battery voltage for handsets 13 and 14:', round(bat_1314['voltage'].mean(), 3), 'V')
print('Average battery voltage for all other handsets: ', round(bat_others['voltage'].mean(), 3), 'V')

---
## FAULT 6: Three Firmware Log Entries Have a Missing Version Value

In [ ]:
# These are separate from the 102 null component rows already fixed
# These rows DO have component = firmware but the version field inside the message is missing
null_fw = fw[fw['version'].isnull()]

print('Firmware records with a missing version value:')
print(null_fw[['ID', 'log_time', 'message']].to_string())

In [ ]:
# Check surrounding firmware entries to infer the likely version at each point
affected_ids = null_fw['ID'].unique()
print('Affected handset IDs:', affected_ids)
print()

for hid in affected_ids:
    fw_for_handset = fw[fw['ID'] == hid].sort_values('log_time')
    print('Firmware history for handset', hid, ':')
    print(fw_for_handset[['log_time', 'version']].to_string())
    print()

---
## Summary of All Faults

| # | Fault | Affected | Severity |
|---|-------|----------|----------|
| 1 | PRC-007 error rate 50%, nearly 4x higher than peers | PRC-007, handsets 13 and 14 | Critical |
| 2 | Handsets 13 and 14 never received firmware 2.0.1 | handset-00013, handset-00014 | Critical |
| 3 | Handset-00005 telemetry stopped after 15 April | handset-00005 at PRC-003 | High |
| 4 | PRC-009 running tests on weekends, outside protocol | PRC-009 | High |
| 5 | Repeated CRITICAL modem and battery alerts on 4 handsets | Handsets 5, 6, 13, 14 | Medium |
| 6 | 3 firmware records with no version value | Handsets 6, 15, 18 | Low |

**Data quality issues resolved before analysis:**
- 33 duplicate rows removed from the hardware table
- 8 rows with empty message bodies dropped
- 102 rows with missing component labels recovered from message content
- Empty status strings noted; voltage and signal values used directly in analysis